In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

# 1 = any illegal drug use, 0 = none
df["drug_use"] = np.where(df["V79"].isna(), np.nan, (df["V79"] > 0).astype(int))

print(df["drug_use"].value_counts(dropna=False))   # expect 3221 zeros, 230 ones, 79 missing

drug_use
0.0    3221
1.0     230
NaN      79
Name: count, dtype: int64


/var/folders/66/ntlk8n594_9346y_qj6fmv0c0000gn/T/ipykernel_55868/4211556772.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["drug_use"] = np.where(df["V79"].isna(), np.nan, (df["V79"] > 0).astype(int))


Codes 2 to 8 have no real labels. Only the two end categories, “Under 15” and “22 or more”, kept their text. The middle ones show their own code number, so “2” doesn’t mean age 2.

They almost certainly stand for single ages. Between “Under 15” and “22 or more”, the 7 codes fit ages 15, 16, 17, 18, 19, 20 and 21 exactly. The counts support it: they peak at codes 4 to 5 (ages 17 to 18), then fall away, as you’d expect for students who are mostly 18 to 23. I’m inferring this, though, so check the ARDA codebook for question 81 before you rely on it.

“Never had intercourse” is code 0, so a non-age answer sits at the bottom of an age scale. Anyone who hasn’t had sex yet looks like the “youngest” starter.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_stata("chrisu.DTA", convert_categoricals=False).copy()
df = df.rename(columns={"V81": "first_sex_code"})

# 1. Ever had sex: 1 = yes, 0 = never, missing stays missing
df["ever_sex"] = np.where(df["first_sex_code"].isna(), np.nan,
                          (df["first_sex_code"] > 0).astype(int))

# 2. Age at first sex, only for those who have had sex (assumed mapping, check the codebook)
age_map = {1: 14, 2: 15, 3: 16, 4: 17, 5: 18, 6: 19, 7: 20, 8: 21, 9: 22}
df["first_sex_age"] = df["first_sex_code"].map(age_map)   # codes 0 and missing become NaN

print(df["ever_sex"].value_counts(dropna=False))
print(df["first_sex_age"].describe())

ever_sex
1.0    2172
0.0    1277
NaN      81
Name: count, dtype: int64
count    2172.000000
mean       16.851289
std         1.539136
min        14.000000
25%        16.000000
50%        17.000000
75%        18.000000
max        22.000000
Name: first_sex_age, dtype: float64


In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_stata("chrisu.DTA", convert_categoricals=False)

df = df.rename(columns={
    "V25": "pray_freq", "V26": "table_grace", "V27": "pray_self",
    "V28": "pray_close", "V29": "pray_others", "V34": "attend_services",
    "V30": "god_close", "V39": "bible_inerrant", "V82": "sex_partners_yr", "V83": "sex_partners_lifetime", "V86": "unprotected_sex_stranger"
})

In [8]:
# 1. Load and rename
df = pd.read_stata("chrisu.DTA", convert_categoricals=False).copy()
df = df.rename(columns={
    # Section A: religious practices
    "V25": "pray_freq", "V26": "table_grace", "V27": "pray_self",
    "V28": "pray_close", "V29": "pray_others", "V34": "attend_services",
    # Section B: religious beliefs
    "V30": "god_close", "V39": "bible_inerrant",
    # Outcome ingredients
    "V81": "first_sex_code", "V82": "sex_partners_yr",
    "V83": "sex_partners_lifetime", "V86": "unprotected_sex_stranger",
})
df["god_close"] = 6 - df["god_close"]       # reverse so higher = closer to God

In [10]:
# 2. Dependent variable: sexual_deviance (1 = any criterion met, 0 = none)
outcome_items = ["first_sex_code", "sex_partners_yr", "sex_partners_lifetime", "unprotected_sex_stranger"]
any_missing = df[outcome_items].isna().any(axis=1)

early_first_sex = df["first_sex_code"].between(1, 3)    # codes 1-3 = under 15, 15, 16 (assumed mapping)
deviant = (early_first_sex |
           (df["sex_partners_yr"] >= 1) |
           (df["sex_partners_lifetime"] >= 1) |
           (df["unprotected_sex_stranger"] >= 0))
df["sexual_deviance"] = np.where(any_missing, np.nan, deviant.astype(int))

In [11]:
# 3. Independent variables, in two sections
practices = ["pray_freq", "table_grace", "pray_self", "pray_close", "pray_others", "attend_services"]
beliefs = ["god_close", "bible_inerrant"]

In [12]:
# 4. One common sample for all models
cb = df[["sexual_deviance"] + practices + beliefs].dropna().copy()
print("N =", len(cb), "| sexual_deviance = 1:", int(cb["sexual_deviance"].sum()))


N = 2987 | sexual_deviance = 1: 2987


In [13]:
m3 = smf.logit("sexual_deviance ~ " + " + ".join(practices + beliefs), data=cb).fit(disp=0)   # both sections

/Users/elavainikrishnamoorthy/Desktop/hs4002_ay26/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/elavainikrishnamoorthy/Desktop/hs4002_ay26/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/elavainikrishnamoorthy/Desktop/hs4002_ay26/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/elavainikrishnamoorthy/Desktop/hs4002_ay26/.venv/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:227: PerfectSepara

In [20]:
print([n for n in dir() if n in ("model", "model1", "m1", "m2", "m3", "practices", "beliefs", "cb")])

['beliefs', 'cb', 'm3', 'practices']


In [25]:
from stargazer.stargazer import Stargazer  
sg = Stargazer([m3])
sg.dependent_variable_name("Sexual Deviance")
sg.custom_columns(["Sexual Deviance"], [1])
sg.covariate_order(practices + beliefs + ["Intercept"])
sg.rename_covariates({
    "pray_freq": "PRACTICES: Praying Frequency", "table_grace": "PRACTICES: Saying Grace",
    "pray_self": "PRACTICES: Praying for Self", "pray_close": "PRACTICES: Praying for Close People",
    "pray_others": "PRACTICES: Praying for Others", "attend_services": "PRACTICES: Attending Services",
    "god_close": "BELIEFS: Closeness to God", "bible_inerrant": "BELIEFS: Bible Inerrancy",
    "Intercept": "Intercept",
})
sg.add_custom_notes(["Standard errors in parentheses."])
sg

In [23]:
print(cb["sexual_deviance"].value_counts(dropna=False))

sexual_deviance
1.0    2987
Name: count, dtype: int64


In [24]:
outcome_items = ["first_sex_code", "sex_partners_yr", "sex_partners_lifetime", "unprotected_sex_stranger"]
any_missing = df[outcome_items].isna().any(axis=1)

deviant = (df["first_sex_code"].between(1, 3) |       # first sex under 15, 15 or 16 (assumed mapping)
           (df["sex_partners_yr"] >= 3) |
           (df["sex_partners_lifetime"] >= 5) |
           (df["unprotected_sex_stranger"] >= 1))

df["sexual_deviance"] = np.where(any_missing, np.nan, deviant.astype(int))

# Safety check: stop with a clear message if the outcome has no variation
counts = df["sexual_deviance"].value_counts()
print(counts)
assert counts.size == 2, "sexual_deviance has only one value, so the logit cannot run"

sexual_deviance
0.0    2123
1.0    1296
Name: count, dtype: int64


In [26]:
practices = ["pray_freq", "table_grace", "pray_self", "pray_close", "pray_others", "attend_services"]
beliefs = ["god_close", "bible_inerrant"]

# Rebuild the sample from the corrected outcome
cb = df[["sexual_deviance"] + practices + beliefs].dropna().copy()
print("N =", len(cb))
print(cb["sexual_deviance"].value_counts())        # should show both 0 and 1

# Refit the model (this replaces the old m3)
m3 = smf.logit("sexual_deviance ~ " + " + ".join(practices + beliefs), data=cb).fit()
print(m3.summary())

N = 2987
sexual_deviance
0.0    1902
1.0    1085
Name: count, dtype: int64
Optimization terminated successfully.
         Current function value: 0.628819
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:        sexual_deviance   No. Observations:                 2987
Model:                          Logit   Df Residuals:                     2978
Method:                           MLE   Df Model:                            8
Date:                Sun, 25 Oct 2026   Pseudo R-squ.:                 0.04035
Time:                        13:41:17   Log-Likelihood:                -1878.3
converged:                       True   LL-Null:                       -1957.3
Covariance Type:            nonrobust   LLR p-value:                 4.270e-30
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.1

In [29]:

# 4. Table in the same style as your screenshot
sg = Stargazer([final_model])
sg.dependent_variable_name("Sexual Deviance")
sg.custom_columns(["Sexual Deviance"], [1])
sg.covariate_order(practices + beliefs + ["Intercept"])
sg.rename_covariates({
    "pray_freq": "PRACTICES: Praying Frequency", "table_grace": "PRACTICES: Saying Grace",
    "pray_self": "PRACTICES: Praying for Self", "pray_close": "PRACTICES: Praying for Close People",
    "pray_others": "PRACTICES: Praying for Others", "attend_services": "PRACTICES: Attending Services",
    "god_close": "BELIEFS: Closeness to God", "bible_inerrant": "BELIEFS: Bible Inerrancy",
    "Intercept": "Intercept",
})
sg.add_custom_notes(["Standard errors in parentheses."])
sg

one unit increase in praying frequency, assoicated  decrease in proababilty of sexual deviance by 0.153 (p<0.01), ceteris paribus.
one unit increase in praying for others, assoicated  decrease in proababilty of sexual deviance by 0.100 (p<0.01), ceteris paribus.
one unit increase in attending services, assoicated  decrease in proababilty of sexual deviance by 0.205 (p<0.01), ceteris paribus. largest coeff value and strongest effect on lower possibility of engagin in sexual deviance.
one unit increase in closeness to god, assoicated  decrease in proababilty of sexual deviance by 0.125 (p<0.01), ceteris paribus.

In [ ]:
# separate binary outcomes (missing stays missing)
def flag(col, condition):
    return np.where(df[col].isna(), np.nan, condition.astype(int))

df["early_first_sex"]      = flag("first_sex_code", df["first_sex_code"].between(1, 3))   # first sex under 15, 15 or 16 (assumed mapping)
df["many_partners_yr"]     = flag("sex_partners_yr", df["sex_partners_yr"] >= 3)
df["many_partners_life"]   = flag("sex_partners_lifetime", df["sex_partners_lifetime"] >= 5)
df["unprotected_stranger"] = flag("unprotected_sex_stranger", df["unprotected_sex_stranger"] >= 1)

outcomes = ["early_first_sex", "many_partners_yr", "many_partners_life", "unprotected_stranger"]
practices = ["pray_freq", "table_grace", "pray_self", "pray_close", "pray_others", "attend_services"]
beliefs = ["god_close", "bible_inerrant"]

# 1 common sample
cb = df[outcomes + practices + beliefs].dropna().copy()
print("N =", len(cb))
print(cb[outcomes].mean().round(3))             # share coded 1 for each outcome

# 1 logit per outcome, same predictors
models = [smf.logit(f"{o} ~ " + " + ".join(practices + beliefs), data=cb).fit(disp=0) for o in outcomes]

#table
sg = Stargazer(models)
sg.dependent_variable_name("Components of Sexual Deviance")
sg.custom_columns(["Early First Sex", "Many Partners (Past Year)", "Many Partners (Lifetime)", "Unprotected Sex with Stranger"],
                  [1, 1, 1, 1])
sg.covariate_order(practices + beliefs + ["Intercept"])
sg.rename_covariates({
    "pray_freq": "PRACTICES: Praying Frequency", "table_grace": "PRACTICES: Saying Grace",
    "pray_self": "PRACTICES: Praying for Self", "pray_close": "PRACTICES: Praying for Close People",
    "pray_others": "PRACTICES: Praying for Others", "attend_services": "PRACTICES: Attending Services",
    "god_close": "BELIEFS: Closeness to God", "bible_inerrant": "BELIEFS: Bible Inerrancy",
    "Intercept": "Intercept",
})
sg.add_custom_notes(["Standard errors in parentheses."])
sg

N = 2987
early_first_sex         0.242
many_partners_yr        0.168
many_partners_life      0.194
unprotected_stranger    0.096
dtype: float64
